# 21 · Convergence with the resolution of the simulation (TNG50-2, -3, -4, the `simulations` sweep)


How much does the resolution of the hydrodynamical simulation matter? TNG50-2, -3 and -4 are the same 35 cMpc/$h$ box
at 8, 64 and 512 times the gas mass of TNG50-1, run with the same model; snapshot 25 is $z = 3$ in all three. The same
$128 \times 128$ lattice $x_i = i\,L/N$ puts the skewers at the **same places** in the three boxes, so the comparison is also
skewer by skewer. The fiducial options (along $y$, 1 km/s, the exact crossings, the integrated profile, the exact
frame; `run_production.py --sweep simulations`). The reference is TNG50-2, the finest available here. Three levels
are few, and the sub-grid model of TNG is calibrated at one resolution and does not converge strictly with it, so the
successive differences (TNG50-4 to -3 to -2) indicate the trend, not a converged answer.


Every run is a folder of `scripts/production/run_production.py` (`<OUT>/<SWEEP>/<run>/run.json`) that names its
lattice in the campaign's skewer store; `load_run` reads it. The statistics follow notebooks 11 and 18 on **one line of
sight** ($y$): the transmission $T = e^{-\tau}$ of every line and of the total, its mean $\bar T$, the overflux
$\delta = T/\bar T - 1$, P1D on its native modes and $P(k, \mu)$ of the total in four wedges
(`summary_stats.run_statistics`). One line of sight gives no ensemble, so the errors come from a **delete-one-block
jackknife**: the lattice is cut into $4 \times 4$ squares of skewers and each sample leaves one square out (the
3D power of the field with that square set to zero, divided by the fraction kept). Every run is compared with the
reference **bin by bin**, as the relative difference $X_{\rm run}/X_{\rm ref} - 1$, with the jackknife error of that
ratio (`compare_runs`): the runs share the skewer positions and the blocks, so the sample variance they share cancels.
The last part gives, per run, **the largest $k$ up to which every bin differs by less than 0.1%, 0.5% and 1%**
(`k_max_table`; P1D on every native mode, $P(k, \mu)$ on every $|k|$ bin).

Colours: the fiducial run of the campaign in black in every notebook; the other runs from light (coarse) to dark (fine)
on one scale shared by notebooks 18–21.


- **Part A — the stored skewers read back:** every run's lattice, pixels, sampling and dataset, and the checks of the
  pairing (one box size, one redshift, the same positions).
- **Part B — the statistics of every run** and their comparison with the reference.
- **Part C — the mean transmission.**
- **Part D — P1D** of every line and of the total. Then the successive ratios TNG50-3/TNG50-4 and TNG50-2/TNG50-3.
- **Part E — the wedges of $P(k, \mu)$** of the total. Then the successive ratios.
- **Part F — the converged range:** the largest $k$ within 0.1%, 0.5% and 1% of the reference, and what it implies.

In [ ]:
import os
from pathlib import Path

# ---- configuration: the sweep (scripts/production/run_production.py --sweep simulations)
SWEEP = "simulations"
RUNS = ["TNG50-4", "TNG50-3", "TNG50-2"]          # coarse to fine
REFERENCE = "TNG50-2"                             # the finest simulation: the reference
FIDUCIAL = "TNG50-2"                              # the fiducial run (black in every figure)
ORDER = ["TNG50-4", "TNG50-3", None, None, "TNG50-2"]   # on the colour scale of the four notebooks: coarsest, ..., finest
RUN_LABELS = {r: r for r in RUNS}
TITLE_STEM = "Simulation resolution"
LINTHRESH = 0.01                                  # [%]
OUT = Path(os.environ.get("SKEWERJAX_PRODUCTION", Path.home() / "skewerjax_production"))   # the --out of the campaign
N_SUB = None                                      # None: each run's lattice; 32: a quick check on the nested 32^2 sub-lattice
N_BLOCKS = 4                                      # jackknife: N_BLOCKS x N_BLOCKS squares of skewers
MU_EDGES = (0.0, 0.5, 0.8, 0.95, 1.0)             # the four wedges of P(k, mu) (notebooks 10 and 11's wedges of xi)
TOLS = (0.001, 0.005, 0.01)                       # the convergence levels of Part F

X_OF_RUN, X_LABEL, X_SCALE = {"TNG50-4": 19250411, "TNG50-3": 152305397}, "gas cells of the simulation", "log"
X_TICKS = {"TNG50-4": "TNG50-4", "TNG50-3": "TNG50-3"}

# ---- imports
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import json, sys, time
import numpy as np
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()                                    # interactive figures in a live notebook
use_style()
from skewerjax.utils.plotting import (add_panel_letters, field_names, run_styles, plot_runs_mean_flux, plot_runs_p1d,
                                      plot_runs_wedges, plot_k_max_summary)
from skewerjax.summary_stats import (LatticeWedges, block_labels, compare_runs, format_k_max_table, k_max_table,
                                     rebin_pixels, run_statistics, set_k_edges)
from skewerjax.utils.reference_data import P1D_DESI_DR1, nearest_reference
from skewerjax.utils.paths import PRODUCTION_DIR, PRODUCTION_SCRIPTS_DIR   # the run folders; the reader of the skewer store
sys.path.insert(0, str(PRODUCTION_SCRIPTS_DIR))
from skewer_store import load_run
from skewerjax.utils.plotting import PALETTE

## Part A — The stored skewers read back

- `OUT`, `SWEEP`, `RUNS`: the run folders `<OUT>/<SWEEP>/<run>`; `REFERENCE` TNG50-2.
- `N_SUB`: `None` reads the $128 \times 128$ lattices; 32 the nested $32 \times 32$ sub-lattices (a quick check).

The checks: the three boxes have the same size and the snapshots the same redshift, so the lattice puts the skewers at
the same comoving positions and the pixels on the same velocities. Then, skewer by skewer, how far the transmissions
of the coarser simulations are from TNG50-2's: the rms of $F_{\rm run} - F_{\rm ref}$ over the pixels and the
correlation coefficient of the two overfluxes, per skewer.

In [ ]:
RECS = {r: json.loads((OUT / SWEEP / r / "run.json").read_text()) for r in RUNS}
S = {r: load_run(OUT / SWEEP / r, N_SUB) for r in RUNS}          # tau (n_lines, N_sk, N_pix), positions, columns, v, dv, L, u
REF = S[REFERENCE]
L, U, PERIOD = REF.L, REF.u, REF.period                          # [cMpc], aH [km/s per cMpc], v_per [km/s]
N = REF.n_per_axis
Z, A_SNAP = RECS[REFERENCE]["snapshot"]["redshift"], RECS[REFERENCE]["snapshot"]["a"]
LINE_IDS = REF.line_ids
FIELDS = (*LINE_IDS, "total")
NAMES = field_names(FIELDS)
print(f"{'run':<13} {'sim':<8} {'lattice':>8} {'N_pix':>6} {'dv [km/s]':>10} {'dx [cMpc]':>10} {'dx [km/s]':>10} {'route':<10} "
      f"{'dataset':<11} {'computed':>8} {'wall':>8}")
for r in RUNS:
    rec, s, sm = RECS[r], S[r], RECS[r]["sampling"]
    print(f"{r:<13} {s.sim:<8} {s.n_per_axis:>5}^2 {s.v.size:>6} {s.dv:>10.4f} {sm['points_cMpc']:>10.3g} {sm['points_kms']:>10.3g} "
          f"{sm['route']:<10} {rec['dataset']['hash']:<11} {rec['timings']['computed']:>8} {rec['wall_s']:>7.0f}s")
    assert np.isfinite(s.tau).all(), r
for r in RUNS:                                    # tau < 0: the round-off of the engine (float32), relative to each skewer's peak
    t = S[r].tau
    neg = t < 0
    if neg.any():
        worst = (-t.min(axis=-1) / t.max(axis=-1)).max(axis=-1)       # (n_lines,) the largest |tau_min| / tau_max of a skewer
        print(f"{r}: tau < 0 in {neg.mean():.2%} of the samples (" + ", ".join(
              f"{NAMES[l]}: min {t[j].min():.2g}, |min| / max per skewer <= {worst[j]:.1e}" for j, l in enumerate(LINE_IDS) if t[j].min() < 0) + ")")
print(f"\nreference {REFERENCE}: {REF.sim} snapshot {REF.snap}, z = {Z:.4f}, L = {L:.3f} cMpc, u = {U:.3f} km/s per cMpc (1 km/s = {1 / U:.5f} cMpc), "
      f"v_per = {PERIOD:.2f} km/s; tau finite in every run")
print("sampling of the reference: " + RECS[REFERENCE]["sampling"]["effect"])

for r in RUNS:
    sn = RECS[r]["snapshot"]
    assert abs(sn["box_cMpc"] - L) < 1e-9 * L and abs(sn["redshift"] - Z) < 1e-9, r
    assert np.array_equal(S[r].positions, REF.positions) and np.array_equal(S[r].v, REF.v), r
print("box size and redshift: " + "; ".join(f"{r}: L = {RECS[r]['snapshot']['box_cMpc']:.6f} cMpc, z = {RECS[r]['snapshot']['redshift']:.6f}, "
                                           f"{RECS[r]['snapshot']['n_gas']:,} gas cells" for r in RUNS))
F_REF = np.exp(-REF.tau)
for r in RUNS:
    if r == REFERENCE:
        continue
    F = np.exp(-S[r].tau)
    rms = np.sqrt(np.mean((F - F_REF) ** 2, axis=-1))                     # (n_lines, N_sk)
    dF, dR = F / F.mean() - 1, F_REF / F_REF.mean() - 1
    rho = np.sum(dF * dR, -1) / np.sqrt(np.sum(dF**2, -1) * np.sum(dR**2, -1))
    print(f"{r} vs {REFERENCE}, per skewer, median [16th, 84th percentile]: " + "; ".join(
        f"{NAMES[l]}: rms(F - F_ref) {np.median(rms[j]):.3g} [{np.percentile(rms[j], 16):.3g}, {np.percentile(rms[j], 84):.3g}], "
        f"correlation {np.median(rho[j]):.3f} [{np.percentile(rho[j], 16):.3f}, {np.percentile(rho[j], 84):.3f}]" for j, l in enumerate(LINE_IDS)))
del F_REF, F

## Part B — The statistics of every run

The 3D bins are set by the lattice: $|k|$ one bin per mode up to $3.5\,k_f$, steps of $0.4\,k_f$
to 2 cMpc$^{-1}$, then 12 logarithmic bins to the transverse Nyquist frequency $\pi N/L$; the four wedges of
`MU_EDGES` in $\mu = k_\parallel/|k|$. P1D stays on its native modes $k_m = 2\pi m/v_{\rm per}$ for the checks, and is
averaged in 40 logarithmic bins for the figures only. Each line gives the time of each step of a run.

Then the successive comparisons: TNG50-3 against TNG50-4 and TNG50-2 against TNG50-3 (the same jackknife blocks).

In [ ]:
KF = 2 * np.pi / L
K_NYQ = np.pi * N / L                                            # the transverse Nyquist frequency of the lattice [1/cMpc]
K_EDGES = set_k_edges(KF, k_1=3.5 * KF, k_2=min(2.0, K_NYQ), k_max=K_NYQ, dk_factor=0.4, n_log=12)
K1D_EDGES = np.geomspace(2 * np.pi / PERIOD, np.pi / min(S[r].dv for r in RUNS), 41)   # display bins of P1D [s/km]
P1D_REF = nearest_reference(P1D_DESI_DR1, Z)
K_DESI = float(np.max(P1D_REF["k"]))                             # [s/km]: the largest k of DESI DR1 at the snapshot's z
GRIDS = {}


def grid_for(n, n_pix):
    """The mu wedges of P on an n x n lattice of n_pix pixels (LatticeWedges: the bins of lattice_grid's P_mu alone, the memory of
    512^2 lattices), cached: the same |k| edges and mu wedges for every run."""
    if (n, n_pix) not in GRIDS:
        GRIDS[n, n_pix] = LatticeWedges(n, n_pix, L, k_edges=K_EDGES, mu_edges=MU_EDGES)
    return GRIDS[n, n_pix]


def statistics(s, transmissions=None, n_pix=None, dv=None):
    """run_statistics of one run (or of ``transmissions`` on ``n_pix`` pixels of ``dv``) on its lattice's jackknife blocks
    and bins, T_bar the mean over every pixel (one scale factor: the box), with the time of each step."""
    n_pix = s.v.size if n_pix is None else n_pix
    steps = []
    out = run_statistics(None if transmissions is not None else s.tau, s.dv if dv is None else dv, line_ids=LINE_IDS,
                         labels=block_labels(s.columns, s.n_per_axis, N_BLOCKS), grid=grid_for(s.n_per_axis, n_pix),
                         transmissions=transmissions, a_pix=np.full(n_pix, A_SNAP), log=steps.append)
    return out, steps


STATS, T0 = {}, time.perf_counter()
for r in RUNS:
    t = time.perf_counter()
    STATS[r], steps = statistics(S[r])
    print(f"{r:<13} {time.perf_counter() - t:6.1f} s: " + "; ".join(steps))
CMP = {r: compare_runs(STATS[r], STATS[REFERENCE], k_edges_1d=K1D_EDGES) for r in RUNS if r != REFERENCE}   # display bins
CMP_ALL = {r: compare_runs(STATS[r], STATS[REFERENCE]) for r in RUNS if r != REFERENCE}                  # every mode: the checks
STYLES = run_styles(ORDER, fiducial=FIDUCIAL)
print(f"every run in {time.perf_counter() - T0:.0f} s; P(k, mu) in {len(MU_EDGES) - 1} wedges x {len(K_EDGES) - 1} |k| bins up to "
      f"{K_NYQ:.2f} cMpc^-1; jackknife over {N_BLOCKS} x {N_BLOCKS} squares of {N // N_BLOCKS} x {N // N_BLOCKS} skewers")

CMP_STEP = {"TNG50-3": compare_runs(STATS["TNG50-3"], STATS["TNG50-4"], k_edges_1d=K1D_EDGES),
            "TNG50-2": compare_runs(STATS["TNG50-2"], STATS["TNG50-3"], k_edges_1d=K1D_EDGES)}
CMP_STEP_ALL = {"TNG50-3": compare_runs(STATS["TNG50-3"], STATS["TNG50-4"]),
                "TNG50-2": compare_runs(STATS["TNG50-2"], STATS["TNG50-3"])}
STEP_LABELS = {"TNG50-4": "TNG50-4", "TNG50-3": "TNG50-3 / TNG50-4", "TNG50-2": "TNG50-2 / TNG50-3"}

## Part C — The mean transmission

$\bar F$ of every line and of the total, per run, as $100\,(\bar F_{\rm run}/\bar F_{\rm ref} - 1)$ [%] with its
jackknife error (the reference is the line at 0; the grey band is $\pm 0.5\%$, the dotted lines $\pm 0.1\%$ and
$\pm 1\%$; the axis is linear within `LINTHRESH`).

In [ ]:
fig, axs = plot_runs_mean_flux(CMP, STYLES, REFERENCE, field_labels=NAMES, labels=RUN_LABELS, linthresh=LINTHRESH,
                               title=f"{TITLE_STEM}: the mean transmission against {RUN_LABELS[REFERENCE]}")
add_panel_letters(axs)
plt.show()
for r, c in CMP_ALL.items():
    print(f"{r:<13} " + "; ".join(f"{NAMES[f]}: {100 * c['mean_flux']['rel'][j]:+.3g} +- {100 * c['mean_flux']['err'][j]:.2g} %"
                                  for j, f in enumerate(FIELDS)))

## Part D — P1D

Above, $k P_{\rm 1D}/\pi$ of the reference (thick, its jackknife error as a band) and of every run; below, the residual
$100\,(P_{\rm run}/P_{\rm ref} - 1)$ [%] with its jackknife error bars, in the same bands as Part C. The grey bands
lie outside $[2\pi/v_{\rm per}, \pi/dv]$ of the reference; the dashed line is the largest $k$ of the DESI DR1 P1D at
the snapshot's redshift. 

In [ ]:
fig, axs = plot_runs_p1d(CMP, STYLES, REFERENCE, field_labels=NAMES, labels=RUN_LABELS, linthresh=LINTHRESH,
                         k_limits=(2 * np.pi / PERIOD, np.pi / REF.dv),
                         title=f"{TITLE_STEM}: $P_{{\\rm 1D}}$ against {RUN_LABELS[REFERENCE]}")
for a in axs.ravel():
    a.axvline(K_DESI, color=PALETTE["grey"], ls="--", lw=1.5, zorder=0)
add_panel_letters(axs[0])
plt.show()

### The successive ratios: TNG50-3/TNG50-4 and TNG50-2/TNG50-3

Above, the three P1D; below, the change at each step of resolution (a factor 8 in mass, 2 in length). If the
simulations converged geometrically, the second step would be smaller than the first by a constant factor $q$ and
the error left in TNG50-2 would be about the second step over $q - 1$; the printout gives $q$ in the DESI range
(the median over the modes of the ratio of the two steps) where both steps exceed their errors.

In [ ]:
fig, axs = plot_runs_p1d(CMP_STEP, STYLES, "TNG50-4", field_labels=NAMES, labels=STEP_LABELS, linthresh=LINTHRESH,
                         k_limits=(2 * np.pi / PERIOD, np.pi / REF.dv),
                         title="Simulation resolution: $P_{\\rm 1D}$ at each step of resolution")
add_panel_letters(axs[0])
plt.show()
k = CMP_STEP_ALL["TNG50-2"]["p1d"]["k"]
for j, f in enumerate(FIELDS):
    d1, e1 = CMP_STEP_ALL["TNG50-3"]["p1d"]["rel"][j], CMP_STEP_ALL["TNG50-3"]["p1d"]["err"][j]
    d2, e2 = CMP_STEP_ALL["TNG50-2"]["p1d"]["rel"][j], CMP_STEP_ALL["TNG50-2"]["p1d"]["err"][j]
    sel = (k <= K_DESI) & (np.abs(d1) > 2 * e1) & (np.abs(d2) > 2 * e2) & (np.sign(d1) == np.sign(d2))
    q = np.median(d1[sel] / d2[sel]) if sel.sum() > 3 else np.nan
    print(f"P1D {NAMES[f]}, k <= {K_DESI:.3f} s/km: median |step| 4->3 {100 * np.median(np.abs(d1[k <= K_DESI])):.2g} %, "
          f"3->2 {100 * np.median(np.abs(d2[k <= K_DESI])):.2g} %; q = step(4->3) / step(3->2) = {q:.2f} on {sel.sum()} modes"
          + (f"; geometric extrapolation of the error left in TNG50-2: ~{100 * np.median(np.abs(d2[sel])) / (q - 1):.2g} %" if q > 1 else ""))

## Part E — The wedges of $P(k, \mu)$

One column per wedge: above, $P(k, \mu)$ of the total overflux of the reference (its jackknife error as a band) and of
every run at the reference's bin centres; below, the residual in per cent as in Part D. 

In [ ]:
fig, axs = plot_runs_wedges(CMP, STYLES, REFERENCE, labels=RUN_LABELS, linthresh=LINTHRESH,
                            title=f"{TITLE_STEM}: $P(k, \\mu)$ of the total against {RUN_LABELS[REFERENCE]}")
add_panel_letters(axs[0])
plt.show()

### The successive ratios of $P(k, \mu)$

In [ ]:
fig, axs = plot_runs_wedges(CMP_STEP, STYLES, "TNG50-4", labels=STEP_LABELS, linthresh=LINTHRESH,
                            title="Simulation resolution: $P(k, \\mu)$ at each step of resolution")
add_panel_letters(axs[0])
plt.show()

## Part F — The converged range

For every run and statistic, the largest $k$ up to which **every** bin differs from the reference by less than 0.1%,
0.5% and 1% (`k_max_table`: P1D on its native modes, $P(k, \mu)$ on its $|k|$ bins; "$\geq k$" when every bin passes up to
the run's last one; 0 when the first bin already fails). The last columns give the run's last bin and the median
jackknife error of the relative difference: a $k_{\max}$ set by bins whose difference is within that error is limited
by noise, not by the parameter. The figure draws $k_{\max}(0.5\%)$ against the number of gas cells of the simulation.

In [ ]:
ROWS = k_max_table(CMP_ALL, TOLS)
print(f"against {REFERENCE}; P1D on its native modes [s/km], P(k, mu) on its |k| bins [1/cMpc]; '>= k': every bin passes "
      f"up to the run's last one; the DESI DR1 P1D reaches k = {K_DESI:.3f} s/km at z = {P1D_REF['z']}")
print(format_k_max_table(ROWS, TOLS))
fig, axs = plot_k_max_summary(ROWS, X_OF_RUN, xlabel=X_LABEL, xscale=X_SCALE, xticks=X_TICKS, tol=0.005, p1d_k_ref=K_DESI,
                              title=f"{TITLE_STEM}: the largest $k$ converged to 0.5% against {RUN_LABELS[REFERENCE]}")
add_panel_letters(axs)
plt.show()

**Reading.** Stated with caution: three resolutions a factor 8 apart in mass, and no rescaling of the mean flux.

* **Mean transmission:** TNG50-4 $-1.09 \pm 0.16$% and TNG50-3 $+1.45 \pm 0.07$% relative to TNG50-2, so it is not
  monotonic with resolution.
* **$\mathrm{Ly}\alpha$ $P_\mathrm{1D}$ within the DESI range:** steps of 7.4% ($-4 \to -3$) and 7.5% ($-3 \to -2$). The steps do
  not shrink ($q \approx 1.2$), so TNG50-2 is not converged at the per-cent level. With $q$ this close to 1 the
  geometric extrapolation printed above ($\sim$40%) is not meaningful.
* **Si III $P_\mathrm{1D}$:** changes by a factor 2–3 per step. It traces dense, enriched gas, which these resolutions
  resolve poorly.
* **$P(k, \mu)$:**
  * TNG50-3 is a few per cent below TNG50-2 up to $\sim$3 cMpc$^{-1}$, and 10–20% below at 7 cMpc$^{-1}$;
  * TNG50-4 is 10–20% above at low $|k|$ in the transverse wedges and up to 50% below at 7 cMpc$^{-1}$;
  * no bin is within 0.5%.
* **Per skewer:** the flux correlates with TNG50-2 at a median 0.985 (TNG50-3) and 0.939 (TNG50-4).

The resolution of the simulation is the largest effect of the four sweeps by orders of magnitude. A comparison of shapes
should first match the mean flux (rescale $\tau$), which this notebook does not do.